We use ast to get statistics about python code like number of lines, number of characters, functions, classes and variables

In [1]:
import ast

class CodeAnalyzer(ast.NodeVisitor):
    def __init__(self, source: str):
        self.source = source
        self.lines = source.splitlines()

        self.empty_lines = 0
        for line in source.splitlines():
            if not line.strip():
                self.empty_lines += 1

        self.functions = []
        self.classes = []
        self.global_vars = set()

    # ---------- Helpers ----------

    def _get_text(self, node):
        if not hasattr(node, "lineno"):
            return ""

        start = node.lineno - 1
        end = node.end_lineno

        return "\n".join(self.lines[start:end])

    def _count_chars(self, node):
        return len(self._get_text(node))

    def _count_lines(self, node):
        return node.end_lineno - node.lineno + 1
    

    # ---------- Variables ----------

    def visit_Assign(self, node):
        for t in node.targets:
            if isinstance(t, ast.Name):
                self.global_vars.add(t.id)

        self.generic_visit(node)

    # ---------- Functions ----------

    def visit_FunctionDef(self, node):
        args = node.args
        arg_names = [a.arg for a in args.args]

        doc = ast.get_docstring(node)
        has_docstring = 1 if doc else 0

        func_info = {
            "name": node.name,
            "lines": self._count_lines(node),
            "chars": self._count_chars(node),
            "arg_names": arg_names,
            "has_docstring": has_docstring
        }

        self.functions.append(func_info)

        self.generic_visit(node)

    # ---------- Classes ----------

    def visit_ClassDef(self, node):
        methods = []
        init_args = []

        doc = ast.get_docstring(node)
        has_docstring = 1 if doc else 0

        for item in node.body:
            if isinstance(item, ast.FunctionDef):

                methods.append(item.name)

                if item.name == "__init__":
                    init_args = [
                        a.arg for a in item.args.args
                        if a.arg != "self"
                    ]

        class_info = {
            "name": node.name,
            "lines": self._count_lines(node),
            "chars": self._count_chars(node),
            "methods": methods,
            "init_args": init_args,
            "has_docstring": has_docstring
        }

        self.classes.append(class_info)

        self.generic_visit(node)

We create a function to help us get information about comments

In [2]:
import tokenize
from io import BytesIO

def extract_comments(code: str) -> int:
    comment_chars = 0

    tokens = tokenize.tokenize(
        BytesIO(code.encode()).readline
    )

    for tok in tokens:
        if tok.type == tokenize.COMMENT:
            comment_chars += len(tok.string)

    return comment_chars

This function wraps up the logic and returns stats as a dict

In [3]:
def analyze_code(code: str) -> dict:
    comment_chars = extract_comments(code)

    stats = CodeAnalyzer(code)
    tree = ast.parse(code)
    stats.visit(tree)

    module_doc = ast.get_docstring(tree)
    has_docstring = 1 if module_doc else 0

    return {
        "lines": len(stats.lines),
        "empty_lines": stats.empty_lines,
        "characters": len(code),
        "comment_chars": comment_chars,
        "has_docstring": has_docstring,
        "functions": stats.functions,
        "classes": stats.classes,
        "variables": stats.global_vars
    }

We use stats to generate features

In [4]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

def generate_features(row: pd.Series) -> dict:
    stats = analyze_code(row["text"])

    # Whole file features
    characters = stats["characters"]
    code_compactness = (stats["lines"] - stats["empty_lines"]) / stats["lines"]
    chars_per_line = stats["characters"] / stats["lines"]
    comment_ratio = stats["comment_chars"] / stats["characters"]
    has_docstring = stats["has_docstring"]

    # Variables features
    num_vars = len(stats["variables"])

    variable_ratio = len(stats["variables"]) / stats["characters"]
    avg_var_name = (sum(len(var) for var in stats["variables"]) / 
                    num_vars) if num_vars else 0

    # Functions and classes features
    num_funcs_and_classes = len(stats["functions"]) + len(stats["classes"])

    avg_func_class_name = ( (sum(len(func["name"]) for func in stats["functions"]) +
                             sum(len(clss["name"]) for clss in stats["classes"])) /
                             num_funcs_and_classes ) if num_funcs_and_classes else 0
    
    avg_func_class_chars = ( (sum(func["chars"] for func in stats["functions"]) +
                              sum(clss["chars"] for clss in stats["classes"])) /
                              num_funcs_and_classes ) if num_funcs_and_classes else 0
    
    avg_func_class_args = ( (sum(len(func["arg_names"]) for func in stats["functions"]) +
                             sum(len(clss["init_args"]) for clss in stats["classes"])) /
                             num_funcs_and_classes ) if num_funcs_and_classes else 0
    
    func_class_docstring_ratio = ( (sum(func["has_docstring"] for func in stats["functions"]) +
                                    sum(clss["has_docstring"] for clss in stats["classes"])) /
                                    num_funcs_and_classes ) if num_funcs_and_classes else 0
    
    return {
        "characters": characters,
        "code_compactness": code_compactness,
        "chars_per_line": chars_per_line,
        "comment_ratio": comment_ratio,
        "has_docstring": has_docstring,
        "variable_ratio": variable_ratio,
        "avg_var_name": avg_var_name,
        "num_funcs_and_classes": num_funcs_and_classes,
        "avg_func_class_name": avg_func_class_name,
        "avg_func_class_chars": avg_func_class_chars,
        "avg_func_class_args": avg_func_class_args,
        "func_class_docstring_ratio": func_class_docstring_ratio
    }

We drop unnecessary columns, generate features for our dataset and save it

In [5]:
df = pd.read_pickle("../data/embeddings_v3.pkl")
df = df.drop(columns=["NUM_CHARS", "pylint_text", "executable_code"])

features = df.progress_apply(generate_features, axis=1, result_type="expand")
df = df.join(features)
df

100%|██████████| 3479/3479 [00:06<00:00, 576.86it/s]


,text,repo_name,path,license,pylint_score,embedding,characters,code_compactness,chars_per_line,comment_ratio,has_docstring,variable_ratio,avg_var_name,num_funcs_and_classes,avg_func_class_name,avg_func_class_chars,avg_func_class_args,func_class_docstring_ratio
0,"""""""\nOutlier Detection using Tukeys Filter Cla...",trademob/anna-molly,lib/plugins/tukeys_filter.py,mit,7.634409,"[-0.38697323, 0.24379842, 0.23942713, -0.03379...",5255.0,0.846154,40.423077,0.039391,1.0,0.004186,7.863636,6.0,6.500000,1654.833333,2.166667,0.000000
1,# Copyright (c) 2018 PaddlePaddle Authors. A...,PaddlePaddle/Paddle,python/paddle/fluid/tests/unittests/xpu/test_r...,apache-2.0,7.037037,"[-0.40957466, 0.19089544, 0.26356944, 0.009104...",6221.0,0.816667,34.561111,0.138724,0.0,0.000482,10.000000,34.0,16.676471,410.058824,0.676471,0.000000
2,# -*- coding: utf-8 -*-\nfrom __future__ impor...,marevol/gym-starter-kit-example,pong/__init__.py,apache-2.0,8.000000,"[-0.2769219, 0.08974697, 0.25448853, -0.019443...",240.0,0.666667,26.666667,0.095833,0.0,0.008333,10.000000,0.0,0.000000,0.000000,0.000000,0.000000
3,import re\n\nfrom tornado.escape import xhtml_...,JmPotato/College,handlers/utils.py,mit,7.872340,"[-0.3845056, 0.09546796, 0.2581636, 0.05189531...",2237.0,0.805970,33.388060,0.000000,0.0,0.005811,7.000000,6.0,12.166667,564.166667,1.166667,0.166667
4,from tests import PMGTestCase\nfrom pmg.models...,Code4SA/pmg-cms-2,tests/unit/test_attendance.py,apache-2.0,8.358209,"[-0.35930574, 0.17806531, 0.24545142, -0.08270...",6441.0,0.893617,34.260638,0.023288,0.0,0.003571,15.782609,6.0,28.166667,2075.500000,0.833333,0.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3474,import time\n\nimport sqlalchemy as SA\n\nimpo...,Yelp/pushmanager,pushmanager/servlets/livepush.py,apache-2.0,7.142857,"[-0.36924505, 0.08506324, 0.24026895, 0.050095...",3789.0,0.900990,37.514851,0.000000,0.0,0.002639,9.100000,4.0,9.250000,1760.500000,1.500000,0.000000
3475,import os\n\nfrom django.core.wsgi import get_...,kaiocesar/easy-bank,easybank/wsgi.py,mit,7.500000,"[-0.4201975, 0.035687566, 0.27887806, 0.156270...",169.0,0.571429,24.142857,0.000000,0.0,0.005917,11.000000,0.0,0.000000,0.000000,0.000000,0.000000
3476,# Copyright 2017 The TensorFlow Authors. All R...,eadgarchen/tensorflow,tensorflow/python/estimator/util_test.py,apache-2.0,0.000000,"[-0.35693988, 0.14048848, 0.25987947, -0.11016...",3897.0,0.703125,30.445312,0.183474,1.0,0.001283,16.000000,19.0,14.894737,363.315789,1.526316,0.000000
3477,from lib.action import BaseVMsAction\n\n__all_...,armab/st2contrib,packs/rackspace/actions/set_vm_metadata_item.py,apache-2.0,6.666667,"[-0.42478552, 0.17662366, 0.28696784, -0.03224...",490.0,0.681818,22.272727,0.000000,0.0,0.008163,5.750000,2.0,10.000000,391.000000,2.500000,0.000000


In [6]:
df.to_pickle("../data/features.pkl")